# 02 · 手刻 Forward Diffusion：加噪的數學

上一課的「線性混入噪聲」只是示意。真正的擴散模型用一套精心設計的**加噪排程**,而且有個漂亮的性質:**任何一步 `x_t` 都能從原圖 `x_0` 一步算出來**,不必真的跑 t 次迴圈。

## 核心公式

> 定義每步的噪聲量 `β_t`,令 `α_t = 1 − β_t`、`ᾱ_t = α_1·α_2···α_t`(連乘)。則:
>
> **x_t = √ᾱ_t · x_0 + √(1−ᾱ_t) · ε** ,其中 ε 是標準高斯噪聲。
>
> t 越大,`ᾱ_t` 越接近 0 → 原圖成分越少、噪聲越多,最後變純雪花。

這個 closed form 是訓練能高效進行的關鍵:隨機挑一個 t,一步就生出對應的 `x_t`。

## 1. 設定

In [ ]:
%pip install -q -U torchvision

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. 建立 beta 排程與 `q_sample`

`alphabars` 就是上面的 ᾱ;`q_sample` 實作那條 closed-form 公式。

In [ ]:
import torch

T = 200                                                  # 擴散步數
betas = torch.linspace(1e-4, 0.02, T, device=device)     # 線性 beta 排程
alphas = 1.0 - betas
alphabars = torch.cumprod(alphas, dim=0)                  # 連乘:一步到位的關鍵


def q_sample(x0, t, noise):
    """前向加噪(closed form):x_t = sqrt(ab)·x0 + sqrt(1-ab)·noise。"""
    ab = alphabars[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise


@torch.no_grad()
def ddpm_sample(model, n=16):
    """反向去噪:從純噪聲一步步還原,共 T 步(慢但標準)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    for i in reversed(range(T)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        a, ab, beta = alphas[i], alphabars[i], betas[i]
        mean = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt()
        x = mean + beta.sqrt() * torch.randn_like(x) if i > 0 else mean
    return x


@torch.no_grad()
def ddim_sample(model, n=16, steps=20):
    """DDIM:確定性取樣,跳著走、只要 steps 步(快很多)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    seq = torch.linspace(T - 1, 0, steps, device=device).long().tolist()
    for j, i in enumerate(seq):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        ab = alphabars[i]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()         # 預測乾淨影像
        if j < len(seq) - 1:
            ab_next = alphabars[seq[j + 1]]
            x = ab_next.sqrt() * x0 + (1 - ab_next).sqrt() * eps
        else:
            x = x0
    return x

In [ ]:
print("擴散步數 T =", T)
print("ᾱ 第一步 =", round(alphabars[0].item(), 4), " 最後一步 =", round(alphabars[-1].item(), 6))
print("→ ᾱ 從接近 1 一路降到接近 0:原圖成分越來越少。")

## 3. 用公式直接生出不同 t 的加噪圖

注意:每張都是**一步算出**的,沒有跑迴圈。

In [ ]:
import torch
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

tf = transforms.Compose([transforms.Resize(32), transforms.ToTensor(),
                         transforms.Normalize((0.5,), (0.5,))])
x0 = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)[1][0]
x0 = x0.unsqueeze(0).to(device)

ts = [0, 25, 50, 100, 150, 199]
fig, axes = plt.subplots(1, len(ts), figsize=(11, 2.2))
for ax, ti in zip(axes, ts):
    t = torch.tensor([ti], device=device)
    xt = q_sample(x0, t, torch.randn_like(x0))     # closed form,一步到位
    ax.imshow(((xt[0, 0].clamp(-1, 1) + 1) / 2).cpu(), cmap="gray")
    ax.set_title(f"t={ti}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 小結

- **前向擴散有 closed form**:`x_t = √ᾱ_t·x_0 + √(1−ᾱ_t)·ε`,任一步一步算出。
- `β` 排程決定加噪速度;`ᾱ_t`(連乘)從 ~1 降到 ~0,圖逐漸變雪花。
- 這個性質讓訓練能「隨機抽一個 t、一步加噪」,不必跑完整鏈條。

下一課:訓練一個 **U-Net** 來學會「看著 x_t,把加進去的噪聲 ε 預測回來」——這就是去噪。